<p align="center">
    <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Vertical-SinFondo.png" width="500"/>
</p>

<h2 align="center"><i>ITESO, Universidad Jesuita de Guadalajara</i></h2>
<h2 align="center"><i>Quantitative Finance</i></h2>
<h2 align="center"><i>Prof. Luis Carlos Alvarado Garnica</i></h2>

# Heston — Implementación Robusta (Sesión 7, final)
---
Tenemos una fórmula de pricing semi-cerrada. Para convertirla en código **robusto** de producción, hay dos familias complementarias de métodos:

| Método de transformada (rápido, exacto) | Monte Carlo (flexible, lento) |
|------------------------------------------|-------------------------------|
| Integración directa / **COS** (Fang-Oosterlee) | Simular las SDEs paso a paso, **esquema QE** (Andersen) |
| Mejor para europeas y **calibración** | Necesario para exóticos / **path-dependent** |

En esta sesión final hacemos ambos robustos: manejar el logaritmo complejo en las transformadas, y manejar $v_t\to 0$ en la simulación. Cerramos con la prueba de fuego: **MC debe converger al precio de la transformada**.


## 0. Librerias

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f9f9f9',
                     'axes.grid':True,'grid.alpha':0.35,'font.size':11,'lines.linewidth':2})
PURPLE='#534AB7'; GREEN='#1D9E75'; ORANGE='#D85A30'; AMBER='#BA7517'; BLUE='#2A7FBF'


## 1. Método 1 — Integración directa (la receta práctica)

Recordamos la fórmula:
$$C = S_t\,P_1 - Ke^{-r\tau}P_2,\qquad P_j=\tfrac12+\tfrac1\pi\int_0^\infty\operatorname{Re}\!\left[\frac{e^{-iu\ln K}\varphi_j(u)}{iu}\right]du$$

**Checklist de robustez (presentación 22):**
1. Usar la forma **Little Trap** ($g_2$) para evitar discontinuidades del logaritmo.
2. Truncar la integral en un $u_{\max}$ finito; el integrando decae como $e^{-cu}$, asi que $u_{\max}\approx 100$-$200$ basta.
3. Usar cuadratura adaptativa (`scipy.integrate.quad`).
4. Cuidar el limite $u\to 0$: empezar en un $\varepsilon$ pequeño.


In [ ]:
def heston_cf_j(u,j,x,v,tau,r,kappa,theta,xi,rho):
    i=1j; uj=0.5 if j==1 else -0.5; bj=kappa-rho*xi if j==1 else kappa
    dj=np.sqrt((rho*xi*i*u-bj)**2+xi**2*(u**2-2*uj*i*u))
    g2j=(bj-rho*xi*i*u+dj)/(bj-rho*xi*i*u-dj)
    Dj=((bj-rho*xi*i*u+dj)/xi**2)*((1-np.exp(dj*tau))/(1-g2j*np.exp(dj*tau)))
    Cj=r*i*u*tau+(kappa*theta/xi**2)*((bj-rho*xi*i*u+dj)*tau-2*np.log((1-g2j*np.exp(dj*tau))/(1-g2j)))
    return np.exp(Cj+Dj*v+i*u*x)

def heston_prob_j(j,x,v,tau,K,r,kappa,theta,xi,rho):
    lnK=np.log(K)
    ig=lambda u:np.real(np.exp(-1j*u*lnK)*heston_cf_j(u,j,x,v,tau,r,kappa,theta,xi,rho)/(1j*u))
    val,_=quad(ig,1e-8,200,limit=200); return 0.5+val/np.pi

def heston_call_direct(S,K,r,tau,v0,kappa,theta,xi,rho):
    x=np.log(S)
    return (S*heston_prob_j(1,x,v0,tau,K,r,kappa,theta,xi,rho)
            - K*np.exp(-r*tau)*heston_prob_j(2,x,v0,tau,K,r,kappa,theta,xi,rho))

# Parametros base
S0,K,r,tau=100.0,100.0,0.02,1.0
v0,kappa,theta,xi,rho=0.04,1.5,0.04,0.5,-0.7
print("Precio (integracion directa):", round(heston_call_direct(S0,K,r,tau,v0,kappa,theta,xi,rho),5))


## 2. Método 2 — El método COS (Fang-Oosterlee, 2008)

El método COS expande la densidad (desconocida) en una **serie coseno de Fourier** sobre un intervalo truncado $[a,b]$, usando la función caracteristica para obtener los coeficientes.

$$C \approx e^{-r\tau}\sum_{k=0}^{N-1}{}' \operatorname{Re}\!\left[\varphi\!\left(\tfrac{k\pi}{b-a}\right)e^{-ik\pi\frac{a}{b-a}}\right]U_k$$

**Por qué COS gana para calibración:** complejidad $O(N)$ con $N\approx 128$-$256$, convergencia **exponencial**, y una sola evaluación de $\varphi$ por término (vectorizable). El rango $[a,b]$ se fija con los **cumulantes** del log-retorno.


In [ ]:
def heston_cf_single(u, x0, v0, tau, r, kappa, theta, xi, rho):
    """CF del log-precio (forma unica, no P1/P2). x0 = ln(S0/K)."""
    i=1j
    d=np.sqrt((rho*xi*i*u-kappa)**2+xi**2*(i*u+u**2))
    g=(kappa-rho*xi*i*u-d)/(kappa-rho*xi*i*u+d)
    C=r*i*u*tau+(kappa*theta/xi**2)*((kappa-rho*xi*i*u-d)*tau-2*np.log((1-g*np.exp(-d*tau))/(1-g)))
    D=((kappa-rho*xi*i*u-d)/xi**2)*((1-np.exp(-d*tau))/(1-g*np.exp(-d*tau)))
    return np.exp(C+D*v0+i*u*x0)

def cos_call(S0,K,r,tau,v0,kappa,theta,xi,rho,N=256,L=12):
    x0=np.log(S0/K)
    # cumulantes 1 y 2 del log-retorno (Fang-Oosterlee / Fabien Le Floc'h)
    c1=r*tau+(1-np.exp(-kappa*tau))*(theta-v0)/(2*kappa)-0.5*theta*tau
    c2=(1.0/(8*kappa**3))*(xi*tau*kappa*np.exp(-kappa*tau)*(v0-theta)*(8*kappa*rho-4*xi)
        +kappa*rho*xi*(1-np.exp(-kappa*tau))*(16*theta-8*v0)
        +2*theta*kappa*tau*(-4*kappa*rho*xi+xi**2+4*kappa**2)
        +xi**2*((theta-2*v0)*np.exp(-2*kappa*tau)+theta*(6*np.exp(-kappa*tau)-7)+2*v0)
        +8*kappa**2*(v0-theta)*(1-np.exp(-kappa*tau)))
    a=c1-L*np.sqrt(abs(c2)); b=c1+L*np.sqrt(abs(c2))
    k=np.arange(N); u=k*np.pi/(b-a)
    # coeficientes del payoff call (chi - psi)
    def chi_psi(c,d):
        kp=k*np.pi/(b-a)
        chi=(1/(1+kp**2))*(np.cos(k*np.pi*(d-a)/(b-a))*np.exp(d)
             -np.cos(k*np.pi*(c-a)/(b-a))*np.exp(c)
             +kp*np.sin(k*np.pi*(d-a)/(b-a))*np.exp(d)
             -kp*np.sin(k*np.pi*(c-a)/(b-a))*np.exp(c))
        psi=np.where(k==0,d-c,
            (np.sin(k*np.pi*(d-a)/(b-a))-np.sin(k*np.pi*(c-a)/(b-a)))*(b-a)/(k*np.pi+1e-30))
        return chi,psi
    chi,psi=chi_psi(0,b)
    Uk=2/(b-a)*(chi-psi)
    cf=heston_cf_single(u,x0,v0,tau,r,kappa,theta,xi,rho)
    terms=np.real(cf*np.exp(-1j*u*a))*Uk
    terms[0]*=0.5
    return K*np.exp(-r*tau)*np.sum(terms)

p_cos=cos_call(S0,K,r,tau,v0,kappa,theta,xi,rho)
p_dir=heston_call_direct(S0,K,r,tau,v0,kappa,theta,xi,rho)
print(f"COS:              {p_cos:.6f}")
print(f"Integr. directa:  {p_dir:.6f}")
print(f"Diferencia:       {abs(p_cos-p_dir):.2e}")


### Convergencia exponencial del COS

Mostramos cómo el error de COS cae exponencialmente al aumentar el número de términos $N$ — una de sus grandes ventajas.


In [ ]:
Ns=[8,16,32,48,64,96,128,192,256]
errs=[abs(cos_call(S0,K,r,tau,v0,kappa,theta,xi,rho,N=n)-p_dir) for n in Ns]

fig,ax=plt.subplots(figsize=(9,5))
ax.semilogy(Ns,errs,'o-',color=PURPLE)
ax.set_xlabel('Numero de terminos N'); ax.set_ylabel('Error absoluto (escala log)')
ax.set_title('Convergencia exponencial del metodo COS')
plt.tight_layout(); plt.show()
print("Con N=128 el error ya es de orden 1e-8: por eso COS es ideal para calibracion.")


## 3. Monte Carlo — por qué Euler ingenuo falla

El enfoque obvio discretiza la SDE de varianza directamente:
$$v_{t+\Delta}=v_t+\kappa(\theta-v_t)\Delta+\xi\sqrt{v_t}\sqrt{\Delta}\,Z$$

**El problema:** el incremento gaussiano puede empujar $v_{t+\Delta}$ a ser **negativo**, y entonces $\sqrt{v_t}$ queda indefinido en el siguiente paso. Esto ocurre siempre que se viola Feller — que es la mayoria de las veces en la práctica. Los parches (absorción $v\to\max(v,0)$, reflexión $v\to|v|$) introducen **sesgo** que no desaparece rápido.


In [ ]:
# Demostracion: contar cuantas veces Euler produce varianza negativa
def euler_negativos(v0,kappa,theta,xi,tau,n_paths=5000,n_steps=100,seed=1):
    rng=np.random.default_rng(seed); dt=tau/n_steps
    v=np.full(n_paths,v0); n_neg=0
    for _ in range(n_steps):
        Z=rng.standard_normal(n_paths)
        v=v+kappa*(theta-v)*dt+xi*np.sqrt(np.maximum(v,0))*np.sqrt(dt)*Z
        n_neg+=np.sum(v<0); v=np.maximum(v,0)  # absorcion para poder seguir
    return n_neg

# Parametros que VIOLAN Feller (xi grande)
xi_viola=0.9
feller=2*kappa*theta>=xi_viola**2
n_neg=euler_negativos(v0,kappa,theta,xi_viola,tau)
print(f"Con xi={xi_viola}: Feller {'cumple' if feller else 'VIOLADA'} "
      f"(2kappa*theta={2*kappa*theta:.3f} vs xi^2={xi_viola**2:.3f})")
print(f"Euler produjo varianza negativa {n_neg} veces (que hubo que absorber, introduciendo sesgo)")


## 4. El esquema QE (Quadratic-Exponential, Andersen 2008)

**Hecho clave:** condicional a $v_t$, la siguiente varianza $v_{t+\Delta}$ sigue una chi-cuadrada no central (exacto). El QE **iguala los primeros dos momentos** de esa distribución usando dos aproximaciones según el régimen.

Momentos condicionales exactos:
$$m=\theta+(v_t-\theta)e^{-\kappa\Delta}$$
$$s^2=\frac{v_t\xi^2 e^{-\kappa\Delta}}{\kappa}(1-e^{-\kappa\Delta})+\frac{\theta\xi^2}{2\kappa}(1-e^{-\kappa\Delta})^2$$

**La regla de switching** con $\psi=s^2/m^2$ y $\psi_c=1.5$:
- $\psi\leq\psi_c$: rama **cuadrática** ($v$ grande) — gaussiana desplazada al cuadrado.
- $\psi>\psi_c$: rama **exponencial** ($v$ pequeño) — exponencial con masa puntual en cero.

El resultado: $v_{t+\Delta}\geq 0$ **por construcción**, sin absorción ni reflexión.


In [ ]:
def heston_mc_qe(S0,K,r,tau,v0,kappa,theta,xi,rho,
                 n_paths=50000,n_steps=100,seed=0,psi_c=1.5):
    """Simulacion de Heston con esquema QE de Andersen. Devuelve (precio, error estandar)."""
    rng=np.random.default_rng(seed)
    dt=tau/n_steps
    lnS=np.full(n_paths,np.log(S0))
    v=np.full(n_paths,v0)
    E=np.exp(-kappa*dt)
    for _ in range(n_steps):
        # momentos condicionales exactos
        m =theta+(v-theta)*E
        s2=(v*xi**2*E/kappa)*(1-E)+(theta*xi**2/(2*kappa))*(1-E)**2
        psi=s2/np.maximum(m**2,1e-12)
        v_new=np.empty_like(v)

        # --- rama cuadratica (psi <= psi_c) ---
        qm=psi<=psi_c
        inv=2/psi[qm]
        b2=inv-1+np.sqrt(inv)*np.sqrt(np.maximum(inv-1,0))
        a=m[qm]/(1+b2)
        Z=rng.standard_normal(qm.sum())
        v_new[qm]=a*(np.sqrt(np.maximum(b2,0))+Z)**2

        # --- rama exponencial (psi > psi_c) ---
        em=~qm
        pe=(psi[em]-1)/(psi[em]+1)
        beta=(1-pe)/np.maximum(m[em],1e-12)
        U=rng.random(em.sum())
        v_new[em]=np.where(U<=pe, 0.0,
                           np.log(np.maximum((1-pe)/np.maximum(1-U,1e-12),1e-12))/beta)

        # --- actualizacion del log-precio con driver correlacionado ---
        Zp=rng.standard_normal(n_paths)
        v_int=0.5*(v+v_new)*dt                      # integral de v por punto medio
        lnS=(lnS + r*dt - 0.5*v_int
             + rho*(v_new-v-kappa*(theta-0.5*(v+v_new))*dt)/xi
             + np.sqrt(np.maximum((1-rho**2)*v_int,0))*Zp)
        v=v_new

    ST=np.exp(lnS)
    payoff=np.maximum(ST-K,0)*np.exp(-r*tau)
    return payoff.mean(), payoff.std()/np.sqrt(n_paths)

p_mc,se=heston_mc_qe(S0,K,r,tau,v0,kappa,theta,xi,rho,n_paths=50000,n_steps=100,seed=42)
print(f"Monte Carlo QE: {p_mc:.4f} +/- {se:.4f}")


## 5. La prueba de fuego: MC vs Transformada

El test ácido de cualquier implementación: el precio Monte Carlo (QE) debe converger al precio de la transformada (COS), dentro del error de Monte Carlo.


In [ ]:
p_cos=cos_call(S0,K,r,tau,v0,kappa,theta,xi,rho)
p_mc,se=heston_mc_qe(S0,K,r,tau,v0,kappa,theta,xi,rho,n_paths=100000,n_steps=200,seed=7)

print(f"COS (exacto):     {p_cos:.4f}")
print(f"Monte Carlo QE:   {p_mc:.4f} +/- {se:.4f}")
print(f"Diferencia:       {abs(p_cos-p_mc):.4f}")
print(f"Intervalo 95%:    [{p_mc-1.96*se:.4f}, {p_mc+1.96*se:.4f}]")
print(f"COS dentro del IC 95%: {'SI' if abs(p_cos-p_mc)<1.96*se else 'NO (aumentar paths/steps)'}")


### Convergencia de Monte Carlo al aumentar los caminos

Visualizamos cómo el precio MC se estrecha alrededor del precio COS conforme aumenta el número de trayectorias.


In [ ]:
path_counts=[2000,5000,10000,25000,50000,100000]
precios=[]; errores=[]
for n in path_counts:
    p,s=heston_mc_qe(S0,K,r,tau,v0,kappa,theta,xi,rho,n_paths=n,n_steps=150,seed=11)
    precios.append(p); errores.append(s)

fig,ax=plt.subplots(figsize=(10,5))
ax.errorbar(path_counts,precios,yerr=1.96*np.array(errores),fmt='o-',
            color=ORANGE,capsize=4,label='MC QE (IC 95%)')
ax.axhline(p_cos,color=BLUE,ls='--',label=f'COS = {p_cos:.4f}')
ax.set_xscale('log'); ax.set_xlabel('Numero de trayectorias')
ax.set_ylabel('Precio del call'); ax.set_title('Convergencia de Monte Carlo QE al precio COS')
ax.legend()
plt.tight_layout(); plt.show()


## 6. Guía de selección de método

| Tarea | Método | Por qué |
|-------|--------|---------|
| Europea vainilla | COS | rápido, exacto, vectorizado |
| Calibración (muchos quotes) | COS | la velocidad es esencial |
| Un precio, robustez | Integral directa | simple, confiable |
| Path-dependent / exótico | QE Monte Carlo | única opción |
| Americana / ejercicio temprano | MC + regresión (LSM) | flexible |

**El flujo pragmático:** calibrar con COS (rápido). Una vez fijos los parámetros, priciar exóticos con QE Monte Carlo. Siempre cruzar-verificar ambos en una vainilla antes de confiar en cualquiera.


## 7. Cierre del módulo de Heston

**El arco completo de las siete sesiones:**
1. Motivamos la volatilidad estocástica desde la sonrisa.
2. Derivamos la PDE de Heston cubriendo dos riesgos.
3. Resolvimos la función caracteristica (Little Trap).
4. Mapeamos parámetros a la superficie.
5. Formulamos la calibración como problema inverso (I y II).
6. Implementamos pricing robusto (COS) y simulación (QE).

De "Black-Scholes está mal" a un pricer de volatilidad estocástica calibrado, validado y de estilo producción — con cada paso explicito y cada número verificado.

> **Ejercicio de cierre sugerido:** toma los parámetros que calibraste en la sesión 6 con datos reales de Yahoo, y usa el esquema QE de este notebook para priciar una opción **asiática** (payoff sobre el promedio del precio) — algo que la fórmula COS no puede hacer, pero Monte Carlo si.
